## Transformation logic

In [0]:
%sql
select *
from workspace.silver.erp_customers
limit 10

customer_id,birthdate,gender
AW00011000,1971-10-06,Male
AW00011001,1976-05-10,Male
AW00011002,1971-02-09,Male
AW00011003,1973-08-14,Female
AW00011004,1979-08-05,Female
AW00011005,1976-08-01,Male
AW00011006,1976-12-02,Female
AW00011007,1969-11-06,Male
AW00011008,1975-07-04,Female
AW00011009,1969-09-29,Male


In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id                     AS customer_id,
    ci.customer_key                    AS customer_number,
    ci.firstname                       AS first_name,
    ci.lastname                        AS last_name,
    la.country                         AS country,
    ci.marital_status                  AS marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END                                AS gender,
    ca.birthdate                      AS birthdate,
    ci.created_date                    AS create_date
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_key = ca.customer_id 
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_key = la.customer_id   
"""

df = spark.sql(query)

## Sanity check of dataframe

In [0]:
df.limit(10).display()

customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date
1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
6,11005,AW00011005,Julio,Ruiz,Australia,Single,Male,1976-08-01,2025-10-06
7,11006,AW00011006,Janet,Alvarez,Australia,Single,Female,1976-12-02,2025-10-06
8,11007,AW00011007,Marco,Mehta,Australia,Married,Male,1969-11-06,2025-10-06
9,11008,AW00011008,Rob,Verhoff,Australia,Single,Female,1975-07-04,2025-10-06
10,11009,AW00011009,Shannon,Carlson,Australia,Single,Male,1969-09-29,2025-10-06


## Write gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

## Sanity check of gold table

In [0]:
%sql
SELECT *
FROM workspace.gold.dim_customers
LIMIT 10

customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date
1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
6,11005,AW00011005,Julio,Ruiz,Australia,Single,Male,1976-08-01,2025-10-06
7,11006,AW00011006,Janet,Alvarez,Australia,Single,Female,1976-12-02,2025-10-06
8,11007,AW00011007,Marco,Mehta,Australia,Married,Male,1969-11-06,2025-10-06
9,11008,AW00011008,Rob,Verhoff,Australia,Single,Female,1975-07-04,2025-10-06
10,11009,AW00011009,Shannon,Carlson,Australia,Single,Male,1969-09-29,2025-10-06
